# Phase 5-C1 — guarded Development execution
DRAFT PR — runtime fixed to an immutable full commit. Actual Drive/Colab preflight is still required; this is not execution approval.
Default PREPARE does not run a strategy or consume a batch. A fresh Runtime must pass PREFLIGHT with the actual Development archive before a separate human EXECUTE decision.
Holdout NO_ACCESS; Formal Market Gate NO_GO. Never retry an existing CLAIM, reservation or uncertain outcome. Keep all evidence for human reconciliation.


In [ ]:
MODE = "PREPARE"  # PREPARE -> personal comments -> PREFLIGHT -> separate human decision
EXECUTION_CODE_SHA = "c51afa6ff2ab2105df5969c00d7a8d4fab72842c"  # Immutable earlier runtime commit; avoids notebook self-reference
GOVERNANCE_SHA = "df56664a12a54762e222f141b50e337c6dfc5088"
CORE_SHA = "50ea3bfa42df7708dd3a1b21f17798c206be054e"
AUTHORIZATION_ID = "RA-IBKR_TSM-SMA520_P5C1-0001"
RECOVERY_ARCHIVE = ""  # Exact reviewed Development recovery archive on Drive
ORIGINAL_PREPARATION_ARCHIVE = ""  # Reviewed original C1 data-pin evidence archive
PREPARATION = ""  # Printed by PREPARE: immutable archive/preparation path
BIND_URL = ""  # Research lead's own BIND_AUTHORIZATION comment
TASK_URL = ""  # Research lead's distinct APPROVE_TASK comment
CONFIRM_AUTHORIZATION = ""  # EXECUTE only: type exact Authorization ID
CONFIRM_REMOTE_CLAIM = False  # EXECUTE only: knowingly create permanent GitHub batch claim ref


In [ ]:
import re, sys, subprocess, pathlib, json, os
if not isinstance(EXECUTION_CODE_SHA, str) or not re.fullmatch(r"[0-9a-f]{40}", EXECUTION_CODE_SHA):
    raise RuntimeError("STOP: this draft needs a reviewed published execution commit; do not substitute the governance SHA")
if MODE not in ("PREPARE", "PREFLIGHT", "EXECUTE"):
    raise RuntimeError("Unsupported mode")
if MODE == "EXECUTE" and (CONFIRM_AUTHORIZATION != AUTHORIZATION_ID or CONFIRM_REMOTE_CLAIM is not True):
    raise RuntimeError("Explicit human execution and remote claim confirmations required")
if not RECOVERY_ARCHIVE or not ORIGINAL_PREPARATION_ARCHIVE:
    raise RuntimeError("Exact reviewed archive paths required")
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.13.4"], check=True)
subprocess.run([sys.executable, "-m", "uv", "python", "install", "3.11.17"], check=True)
subprocess.run([sys.executable,"-m","uv","venv","--seed","--python","3.11.17","/content/c1-python"],check=True)
PYTHON = "/content/c1-python/bin/python"
assert subprocess.check_output([PYTHON,"-c","import sys; print(sys.version_info[:2])"],text=True).strip() == "(3, 11)"


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")  # Never print or save credentials
BASE = pathlib.Path("/content/c1-execution-runtime")
BASE.mkdir(exist_ok=False)  # Re-run requires a fresh Runtime; no automatic retry

def checkout(repository, sha, name):
    target = BASE / name
    subprocess.run(["git", "clone", "https://github.com/hh4832/"+repository+".git",str(target)],check=True)
    subprocess.run(["git","-C",str(target),"checkout","--detach",sha],check=True)
    assert subprocess.check_output(["git","-C",str(target),"rev-parse","HEAD"],text=True).strip()==sha
    return target
CODE = checkout("strategy-research-agent", EXECUTION_CODE_SHA, "code")
GOV = checkout("strategy-research-agent", GOVERNANCE_SHA, "governance")
CORE = checkout("market-research-core", CORE_SHA, "core")
subprocess.run([PYTHON,"-m","pip","install","-e",str(CORE),"-e",str(CODE)+"[research]","pyarrow"],check=True)


In [ ]:
settings = dict(mode=MODE, code_sha=EXECUTION_CODE_SHA, governance_root=str(GOV),
    drive_root="/content/drive/MyDrive/Quant_Research/strategy-research-agent/phase5c1_execution",
    recovery_archive=RECOVERY_ARCHIVE, original_preparation_archive=ORIGINAL_PREPARATION_ARCHIVE,
    preparation=PREPARATION, bind_url=BIND_URL, task_url=TASK_URL,
    confirm_authorization=CONFIRM_AUTHORIZATION, confirm_remote_claim=CONFIRM_REMOTE_CLAIM)
SETTINGS = BASE / "settings.json"  # No credentials
SETTINGS.write_text(json.dumps(settings))
subprocess.run([PYTHON,"-m","strategy_agent.execution.c1_runner","--settings",str(SETTINGS)],check=True,cwd=CODE)


## Human sequence
1. After publication, fill the reviewed full execution SHA and exact Drive archive paths. In a fresh Runtime run PREPARE. Preserve its immutable archive and printed request/task hashes. No batch is consumed.
2. Review the artifacts. Post the printed BIND_AUTHORIZATION and APPROVE_TASK lines personally using the existing governance process. This notebook never posts comments.
3. In another fresh Runtime select PREFLIGHT, use the same published SHA and archived PREPARATION path, and supply the two real comment URLs. Require PASS from actual receipt renewal, PIT, all four dated costs, live governance/claim checks and backend identity. This is not a strategy run.
4. Only after that evidence is reviewed may the research lead decide on one EXECUTE attempt, separately setting both confirmations. GitHub Contents/Read and Git refs/Write access as hh4832 is required. The permanent research-claims ref is the server lock; never delete it to retry.
5. Any interruption, prior claim/reservation, uncertain settlement or incomplete archive means STOP for manual reconciliation. Do not rerun EXECUTE. An archive is append-only and hash verified, not Google Drive WORM storage; external modifications are detectable but physical immutability is not guaranteed by Drive.
